<div class="alert alert-block alert-info">
<b>✅ NOTA</b>: Este notebook se ha validado con un entorno creado en Anaconda con Python 3.12.X.
</div>

# _COOL_ (CLIPS Object Oriented Programming Language) con CLIPSPY 1.0.5

Basado en los <a href="http://luisguillermo.com/Curso-IA-UPB/clips-castellano.pdf"> apuntes</a> encontrados en LuisGuillermo.com sobre el curso de IA de la UPB de Aitor San Juan Sánchez (aitorsj@hotmail.com). 

## Instalar [`clipspy`](https://pypi.org/project/clipspy/)

In [1]:
try:
    import clips
except ImportError as e:
    !pip install clipspy==1.0.5
    import clips

In [2]:
import clips

env = clips.Environment()

## Definición de Clases - ***defclass***

`(defclass <nombre> [<comentario>]`  
`   (is-a <nombre-de-superclase>+)`   
`   [(role concrete | abstract)]`   
`   [(pattern-match reactive | non-reactive)]`  
`    <slot>* ;;; definición de los atributos de la clase `       
`    <documentación-handler>*  `

`)`

## _Facets_ - Propiedades de los atributos (_slots_)

Las _facets_ son las propiedades de los atributos, las cuales son ciertas para todos los objetos que comparten dichos _slots_. Las posibles _facets_ son: ***default***, ***storage***, ***access***, ***propagation***, ***source***, ***pattern-match***, ***visibility***, ***create-accessor***, ***override-message*** y ***constraint***.

### _default_ y _default-dynamic_

`(default ?DERIVE | ?NONE | <expresión>*) `

`(default-dynamic <expresión>*)`

* La propiedad `default` es estática, esto es, la expresión se evalúa una sola vez cuando se define la clase y a continuación se almacena para la clase. Por defecto, se supone `?DERIVE`, que hace que el valor del atributo se obtenga por derivación a partir de sus propiedades. Si se especifica `?NONE`, no se asigna ningún valor por defecto al atributo, y se requerirá uno al crear las instancias.
* La propiedad `default-dynamic` es dinámica, esto es, la expresión dada se evalúa cada vez que se crea una instancia, y luego se almacena en el atributo.

In [3]:
camaraClass = '''(defclass camara (is-a USER) (role concrete)
                    (slot numero-serie (default-dynamic (gensym))) 
                    (slot cargada (default no))
                    (slot tapa-cierre (default cerrada))
                    (slot seguro (default puesto))
                    (slot num-fotos-sacadas (default 0)))'''

In [4]:
env.build(camaraClass)

In [5]:
defclass = env.find_class('camara')
instMiCamara = defclass.make_instance('mi-camara')
print('Slot value: seguro - ', instMiCamara['seguro'])

instMiCamara.modify_slots(seguro=clips.Symbol('quitado'))

print('Slot value: seguro - ', instMiCamara['seguro'])


Slot value: seguro -  puesto
Slot value: seguro -  quitado


In [6]:
for inst in env.instances():
    print(inst)
    print('Slot value: ', inst['cargada'])
    print(dict(inst))

[mi-camara] of camara (numero-serie gen1) (cargada no) (tapa-cierre cerrada) (seguro quitado) (num-fotos-sacadas 0)
Slot value:  no
{'numero-serie': 'gen1', 'cargada': 'no', 'tapa-cierre': 'cerrada', 'seguro': 'quitado', 'num-fotos-sacadas': 0}


### _storage_

`(storage local | shared)`

* `local`: el valor del atributo se almacena en la instancia (por defecto). Cada instancia tiene su propio valor del atributo.
* `shared`: el valor del atributo se almacena en la propia clase. Todas las instancias tendrán el mismo valor para el atributo. Si se modifica, se cambiará para todas las instancias.

In [7]:
sVolvo = '''(defclass camion-volvo (is-a USER) 
                (role concrete)
                (slot motor  (create-accessor write)
                             (storage shared)
                             (default v207))
                (slot ruedas (create-accessor write)
                             (storage shared)
                             (default-dynamic 10))
                (slot carga  (create-accessor write)
                             (storage local))
                )
                '''

In [8]:
env.build(sVolvo)

In [9]:
instCamion1 = env.find_class('camion-volvo').make_instance('camion1')

In [10]:
for inst in env.instances():
    print(inst)

[mi-camara] of camara (numero-serie gen1) (cargada no) (tapa-cierre cerrada) (seguro quitado) (num-fotos-sacadas 0)
[camion1] of camion-volvo (motor v207) (ruedas 10) (carga nil)


In [11]:
print(instCamion1)

[camion1] of camion-volvo (motor v207) (ruedas 10) (carga nil)


In [12]:
m = '(modify-instance [' + instCamion1.name + '] (motor v300))'
env.eval(m)
print(instCamion1)

[camion1] of camion-volvo (motor v300) (ruedas 10) (carga nil)


### _access_ - Tipo de acceso permitido sobre un _slot_

`(access read-write | read-only | initialize-only)`

* `read-write`: el _slot_ puede ser leído y modificado (por defecto).
* `read-only` : el _slot_ sólo puede ser leído.
* `initialize-only` : el _slot_ sólo puede ser leído, aunque puede recibir un valor en la creación de la instancia.

In [13]:
sVolvo = '''(defclass camion-volvo (is-a USER) 
                (role concrete)
                (slot motor  (access read-only)
                             (default v350))
                (slot ruedas (create-accessor write)
                             (access initialize-only))
                (slot carga  (create-accessor write)
                             (access read-write))
                )
                '''

In [14]:
env = clips.Environment()
###env.define_function(print)

env.build(sVolvo)

In [15]:
sMess = '''(defmessage-handler camion-volvo put-motor (?valor))'''

In [16]:
env.build(sMess)

In [17]:
instCamion1 = env.find_class('camion-volvo').make_instance('camion1')

### _propagation_ - Heredabilidad de un _slot_

`(propagation inherit | no-inherit)`

* `ìnherit`: se hereda el _slot_ por subclases (por defecto).
* `no-inherit`: no se hereda el _slot_ por subclases.

In [19]:
sDep = '''(defclass deposito (is-a USER) 
                (role concrete)
                (slot capacidad (propagation inherit))
                (slot estado (propagation no-inherit))
                (slot contiene-litros (propagation inherit))
                )
            '''
sDepV = '''(defclass deposito-viejo (is-a deposito))'''

In [20]:
env.build(sDep)
env.build(sDepV)

In [21]:
instDep1 = env.find_class('deposito').make_instance('deposito1')
instDepV1 = env.find_class('deposito-viejo').make_instance('deposito-viejo1')

In [22]:
print(instDep1)
print(instDepV1)

[deposito1] of deposito (capacidad nil) (estado nil) (contiene-litros nil)
[deposito-viejo1] of deposito-viejo (capacidad nil) (contiene-litros nil)


### _source_

`(source exclusive | composite)`

* `exclusive`: (origen de otras facets) al obtener los atributos de la lista de precedencia de clases durante la creación de una instancia, por defecto se toman las facets de la clase más específica y se dan los valores por defecto a las facets no especificadas. Los valores para esta propiedad son: exclusive y composite. El valor exclusive corresponde al comportamiento por defecto explicado anteriormente. 
* `composite`: es que las propiedades no especificadas explícitamente por la clase más específica se toman de la siguiente clase más específica.

### _pattern-match_

`(pattern-match reactive | non-reactive)`

* `reactive`: Valor por defecto. Cualquier cambio en un atributo de una instancia se considerará por el proceso de _pattern-matching_.
* `non-reactive`: No se consideran los cambios.

In [23]:
sPattMatch_0 = '''(defclass camión (is-a USER) 
                        (role concrete)
                        (pattern-match reactive)
                        (slot carga (create-accessor write)
                                    (pattern-match reactive))                
                    )
            '''

El _pattern-match_ del atributo *carga* no es necesario, pues el primer _pattern- match_, al no ir asociado a ningún atributo concreto, se asociará con todos los atributos de la clase.

In [24]:
sPattMatch_1 = '''(defclass coche (is-a USER) 
                        (role concrete)
                        (pattern-match reactive)
                        (slot carga (create-accessor write)
                                    (pattern-match non-reactive))
                    )
            '''

Modificaciones en el atributo *carga* de la clase *coche* no provocarán cambios en la agenda (base de reglas).

<div class="alert alert-block alert-danger">
<b>NOTA</b>: <tt>Jupyter</tt> captura la salida estándar y la envía al <tt>HTML</tt>, pero no hace lo mismo con <tt>CLIPS</tt>, con lo que hay que hacerlo <b>"a mano"</b>. <tt>CLIPS</tt> usa <b>routers</b> para redirigir la E/S, y podemos usar el <tt>LoggingRouter</tt> que incluye <tt>clipspy</tt> para ello y así conseguir que pueda funcionar <tt>printout t</tt> para sacar mensajes dentro del propio <b>notebook</b>. 
</div>

In [25]:
import logging
import clips

logging.basicConfig(level = 10, format = '%(message)s')

env = clips.Environment()

router = clips.LoggingRouter()
env.add_router(router)

In [26]:
env.eval('(printout t "Hello World!" crlf)')

Hello World!


In [27]:
env.build(sPattMatch_0)
env.build(sPattMatch_1)

In [28]:
sRuleCrear = '''(defrule crear
                    ?instancia <- (object (is-a coche | camión))
                    =>
                    (printout t "Crear " (instance-name ?instancia) crlf)
                )
                '''
sRuleAcceso = '''(defrule acceso-carga
                    ?instancia <- (object (is-a coche | camión) (carga ?)) 
                    =>
                    (printout t "Acceso-carga " (instance-name ?instancia) crlf)
                )
                '''

In [29]:
env.build(sRuleCrear)
env.build(sRuleAcceso)
env.reset()

In [30]:
for rule in env.rules(): 
    print(rule)

(defrule MAIN::crear ?instancia <- (object (is-a coche|camión)) => (printout t "Crear " (instance-name ?instancia) crlf))
(defrule MAIN::acceso-carga ?instancia <- (object (is-a coche|camión) (carga ?)) => (printout t "Acceso-carga " (instance-name ?instancia) crlf))


In [31]:
instCam1 = env.find_class('camión').make_instance('camion1')
instCoc1 = env.find_class('coche').make_instance('coche1')

In [32]:
for ins in env.instances():
    print(ins)

[camion1] of camión (carga nil)
[coche1] of coche (carga nil)


In [33]:
env.run()

Crear [coche1]
Crear [camion1]
Acceso-carga [camion1]


3

### _visibility_

`(visibility private | public)`

* `private`: Valor por defecto. Sólo pueden acceder a los atributos definidos en una clase los gestores de mensajes ligados a la propia clase.
* `public`: Se permite dicho acceso a los gestores de mensajes de superclases o subclases que heredan el atributo.

In [34]:
sPersona = '''(defclass persona (is-a USER)
                    (slot edad (visibility private))
                )
            '''

El atributo _edad_ no puede ser accedido por gestores que no estén asociados a la clase persona.

In [35]:
sHombre = '''(defclass hombre (is-a persona) 
                (role concrete)
                )
            '''

**CLIPS** no permitirá definir un gestor asociado a la clase _hombre_ que intente acceder al atributo _edad_ de la clase _persona_. Si se intenta, se producirá un error.

### _create-accessor_

`(create-accessor ?NONE | read | write | read-write)`

El usuario debe definir explícitamente los gestores de mensajes necesarios para acceder (leer y/o escribir) a cada atributo. Esta propiedad hace que **CLIPS** los cree automáticamente. 
Por defecto, no se crean. 
Además, estos gestores de mensajes no pueden ser modificados por el usuario.

In [36]:
sPersona2 = '''(defclass persona (is-a USER) 
                    (slot concrete)
                    (slot empresa (create-accessor write)) 
                    (slot edad)
                )
            '''

Si al crear una instancia se intenta dar un valor inicial al atributo _edad_, el sistema generará un error porque no existe ningún gestor asociado a la modificación de este atributo. Con esta definición, es imposible acceder al atributo _edad_.

### _override-message_

`(override-message ?DEFAULT | <nombre-mensaje>)`

Permite al usuario definir sus propios mensajes de acceso a los atributos, de forma que puedan ser utilizados directamente por las funciones **COOL**. Por defecto, estos mensajes tienen la sintaxis `put-<nombre-slot>`. Con esta propiedad, se puede definir otro formato distinto.

In [37]:
sPersona3 = '''(defclass persona (is-a USER) 
                    (role concrete)
                    (slot nombre (override-message nombre-put))
                )
            '''

El mensaje que se debe usar para modificar el atributo nombre debe ser `nombre-put`.

### _constraint_

**COOL** permite una serie de restricciones sobre los atributos (tipo, rango, cardinalidad) que pueden ser comprobadas de forma estática (por defecto) y/o de forma dinámica sobre las clases y sus instancias.

* comprobación estática: Valor por defecto. Se realiza cuando se analizan (parse) construcciones o comandos que especifican información sobre atributos.
* comprobación dinámica: se comprueban los valores de nuevas instancias cada vez que se modifican. Activación de comprobación de restricciones dinámica: `(set-dynamic-constraint-checking TRUE)`

In [38]:
sPersona4 = '''(defclass persona (is-a USER) 
                    (role concrete)
                    (multislot empleos (create-accessor write) (type SYMBOL)
                                            (cardinality 0 2))
                )
                '''

In [39]:
env.build(sPersona4)
env.reset()

In [40]:
sDynamic = '''(set-dynamic-constraint-checking TRUE)'''

In [41]:
env.eval(sDynamic)

'FALSE'

In [42]:
env.find_class('persona').make_instance('Ana')

Instance: [Ana] of persona (empleos)

In [43]:
env.find_class('persona').make_instance('Maria', empleos = [clips.Symbol('medico')])

Instance: [Maria] of persona (empleos medico)

In [44]:
instAna2 = env.find_instance('Ana')
m = '(modify-instance [' + instAna2.name + '] (empleos 1.5 medico))'
env.eval(m)
print(instAna2)

[CSTRNCHK1] The value (1.5 medico) for slot 'empleos' of instance [Ana] found in modify-instance does not match the allowed types.
[PRCCODE4] Execution halted during the actions of message-handler 'direct-modify' primary in class 'USER'


CLIPSError: [CSTRNCHK1] The value (1.5 medico) for slot 'empleos' of instance [Ana] found in modify-instance does not match the allowed types. [PRCCODE4] Execution halted during the actions of message-handler 'direct-modify' primary in class 'USER'

In [45]:
for ins in env.instances():
    print(ins)

[Ana] of persona (empleos)
[Maria] of persona (empleos medico)


In [46]:
env.find_class('persona').make_instance('Pepe', empleos = [clips.Symbol('actor'), clips.Symbol('poeta'), clips.Symbol('piloto')])

[CSTRNCHK1] The value (actor poeta piloto) for slot 'empleos' of instance [Pepe] found in InstanceBuilder call does not satisfy the cardinality restrictions.


CLIPSError: [CSTRNCHK1] The value (actor poeta piloto) for slot 'empleos' of instance [Pepe] found in InstanceBuilder call does not satisfy the cardinality restrictions.

Falla porque estamos superando la cardinalidad permitida del atributo _empleos_.

## Proceso de _pattern-matching_ con objetos

Al igual que los hechos, las instancias de clases definidas por el usuario también pueden hacerse corresponder (hacer _matching_) con patrones en la LHS de las reglas. Estos patrones solamente pueden emparejarse (_matchear_) con objetos (instancias) de clases ya existentes y que estén en el ámbito (es decir, que sean visibles) del módulo actual.
Las modificaciones en atributos no reactivos o en instancias de clases no reactivas no tendrán efecto alguno en las reglas.

`(object <atributo-de-restricción>) `

`<atributo-de-restricción> ::= `

`            (is-a <restricción>) | `

`            (name <restricción>) | `

`            (<nombre-atributo> <restricción>*)`

* `is-a`: se utiliza para especificar restricciones de clase del estilo “¿Es este objeto miembro de la clase C?”
* `name`: se utiliza para referirse a una instancia específica, concreta con un nombre determinado, esto es, contiene el nombre de la instancia. La evaluación de este atributo debe ser de tipo `INSTANCE-NAME`, no un símbolo. El nombre de una instancia va encerrado entre corchetes (`[]`). No se pueden usar restricciones multicampo (es decir, el símbolo `$?`) con `is-a` o `name`.

In [47]:
sRuleUnidadControl = '''(defrule UNIDAD-CONTROL::UC7
                            "Apaga la caldera si la habitación está vacía" 
                            (object (is-a Termostato) (temperatura ?t)
                                    (temperatura-seleccionada ?ts & :(>= ?t (- ?ts 3)))
                                    (caldera ?c) (modo calor) (habitación ?h)) 
                            ?inst <- (object (is-a Caldera) (name ?c)
                            (estado encendida))
                            (object (is-a Habitación) (name ?h) (ocupación vacía))
                            =>
                            (modify-instance ?inst (estado apagada)))'''

## Cómo generar automáticamente nombres de instancia

En ocasiones queremos crear instancias en reglas y queremos que su nombre sea distinto del de otras que ya tenemos. Para ello, podemos utilizar las funciones:

* `gensym`: genera un símbolo del estilo `genN` donde `N` es un número.
* `gensym*`: hace lo mismo, pero asegurándose de que ese símbolo no exista ya. 

Se puede reiniciar el contador que usan estas dos funciones con el operador 
`(setgen <num>)` donde `<num>` es el número por el que queremos que empiecen ahora los símbolos.

In [48]:
sInst_Persona = """(defrule Crear_Persona
                        =>
                        (make-instance (gensym) of persona)
                    )"""
env.build(sInst_Persona)

Cada vez que se ejecute se generará un nombre de instancia nuevo.

Si creamos instancias de diferentes clases y queremos que las instancias tengan nombres que podamos identificar podemos concatenar símbolos a los símbolos generados por gensym.

In [49]:
sInst_Persona2 = """(defrule Crear_Persona2
                        =>
                        (make-instance (sym-cat pers- (gensym)) of persona)
                    )"""
env.build(sInst_Persona2)

In [50]:
env.run()

2

In [51]:
for ins in env.instances():
    print(ins)

[Ana] of persona (empleos)
[Maria] of persona (empleos medico)
[pers-gen1] of persona (empleos)
[gen2] of persona (empleos)


## Cómo guardar en fichero los datos de un entorno

Se guardan por separado las clases y las instancias:

In [52]:
env.save("cool_classes.txt")
env.save_instances("cool_instances.txt")

4

Se cargan también por separado:

In [53]:
prueba_env = clips.Environment()

prueba_env.load("cool_classes.txt")
prueba_env.load_instances("cool_instances.txt")

[INSFILE2] File 'cool_instances.txt' is not a binary instances file.


4